In [ ]:
from pathlib import Path
from typing import Annotated, Literal
from datetime import datetime
from copy import copy
import math

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import PowerNorm
from matplotlib.patches import Rectangle, Patch
import matplotlib.gridspec as gridspec

import matplotlib as mpl
import seaborn as sns

import geopandas
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs
import matplotlib as mpl
from mpl_sankey import sankey

from IPython.display import display, HTML, Markdown

from climate_health_map.shared import get_logger
from climate_health_map.data.labels import LABELS, Collection, Group, AggTopic, AggAggTopic
from climate_health_map.data.geographies import load_country_infos, load_grid_data, load_annual_population, merge_grid_info, LOCATION_FEATURE_CODES
from climate_health_map.data.export.lancet.loaders import read_base_data
from climate_health_map.data.export.utils import label_group_counts
from climate_health_map.data.export.lancet.regional_groups import region_groups

logger = get_logger('notebook', loglevel='INFO')

In [ ]:
OUTPUT_DIR = Path('../data/exports/lancet/Global/Figures')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Load data

In [ ]:
try:
    shapes = geopandas.read_file('../data/shapes_2026/WB_GAD_ADM0_complete.shp', encoding='utf-8')
except:
    logger.warning('Official World Bank shapefile missing, falling back to natural_earth default')
    shpfilename = shpreader.natural_earth(
        resolution='10m',
        category='cultural',
        name='admin_0_countries'
    )
    shapes = geopandas.read_file(shpfilename, encoding='utf-8')

# See https://cartopy.readthedocs.io/stable/reference/projections.html

PROJECTION = ccrs.PlateCarree  
AXIS_PROJECTION = ccrs.EqualEarth # Lancert author guideline wants Equal Earth
# #EckertIII

In [ ]:
df, df_locations, df_locations_flat, location_groups, df_affiliations, df_affiliations_flat, affiliation_groups = read_base_data(
    source=Path('../data/exports/20260309'),
    year_start=2006,
    year_end=2025,
    filter_mai=True,
    filter_rel=True,
    prefilter_affiliations=True,
    prefilter_locations=True,
    fix_locations=True,
    logger=logger.getChild('reader'),
)
del df_affiliations_flat, df_locations_flat

In [ ]:
df_countries = load_country_infos()
df_grid = load_grid_data()
df_population = load_annual_population()
df_population = df_population[(df_population['year'] >= df['Publication year'].min()) & (df_population['year'] <= df['Publication year'].max())]

In [ ]:
# Match grid info to locations and items
# This is a bit more complex, because some locations need to be projected to a shape (e.g. country) and then back to grid cells
df, df_locations, grid_counts = merge_grid_info(
    df=df,
    df_locations=df_locations,
    df_grid=df_grid,
    df_population=df_population,
    shapes=shapes,
    location_codes_abstracted=LOCATION_FEATURE_CODES[0],
    location_codes_direct=LOCATION_FEATURE_CODES[3] | LOCATION_FEATURE_CODES[2] | LOCATION_FEATURE_CODES[1],
    logger=logger,
    reference_year=2024,  # We propbably want to use the latest available population count for normalisation
    resolution=2.5,
)

In [ ]:
ORDERING = {
    key.replace('Affiliation_', '').replace('Location_', ''): [li.name for li in label.labels]
    for key, label in LABELS.items()
    if 'topic' not in key and label.type in {'single', 'multi'} and len(label.labels)>0
}
df['incl_py'] = (df['Publication year'] >= 2006) & (df['Publication year'] <= 2025) & df['Publication year'].notna()

In [ ]:
reference_counts = pd.DataFrame(
{1990: {'CC': 2443, 'OA': 2524197}, 1991: {'CC': 2673, 'OA': 2491630}, 1992: {'CC': 3334, 'OA': 2639617}, 1993: {'CC': 3839, 'OA': 2739265}, 1994: {'CC': 3882, 'OA': 2891965}, 1995: {'CC': 4505, 'OA': 3099382}, 1996: {'CC': 4993, 'OA': 3333659}, 1997: {'CC': 5297, 'OA': 3406883}, 1998: {'CC': 5995, 'OA': 3627485}, 1999: {'CC': 6749, 'OA': 3788161}, 2000: {'CC': 8587, 'OA': 4713987}, 2001: {'CC': 10050, 'OA': 4546253}, 2002: {'CC': 10661, 'OA': 5227380}, 2003: {'CC': 12527, 'OA': 5834771}, 2004: {'CC': 13129, 'OA': 6076013}, 2005: {'CC': 17171, 'OA': 6652061}, 2006: {'CC': 18614, 'OA': 10742643}, 2007: {'CC': 25208, 'OA': 7979483}, 2008: {'CC': 32805, 'OA': 8699006}, 2009: {'CC': 43565, 'OA': 9256871}, 2010: {'CC': 51025, 'OA': 10047169}, 2011: {'CC': 59002, 'OA': 11113671}, 2012: {'CC': 66013, 'OA': 12208177}, 2013: {'CC': 76009, 'OA': 12589857}, 2014: {'CC': 81529, 'OA': 14153304}, 2015: {'CC': 95166, 'OA': 15746908}, 2016: {'CC': 92903, 'OA': 16275273}, 2017: {'CC': 117059, 'OA': 15153007}, 2018: {'CC': 99480, 'OA': 14769957}, 2019: {'CC': 115752, 'OA': 16800535}, 2020: {'CC': 132747, 'OA': 16392843}, 2021: {'CC': 161477, 'OA': 17540298}, 2022: {'CC': 183028, 'OA': 16881238}, 2023: {'CC': 210592, 'OA': 21932207}, 2024: {'CC': 231969, 'OA': 29546359}, 2025: {'CC': 247091, 'OA': 45230432}, 2026: {'CC': 51694, 'OA': 11525620}}
).T.rename_axis('Publication year')
display(reference_counts.T)
fig, ax = plt.subplots(figsize=(8, 3))
reference_counts.loc[2000:2025]['CC'].plot(ax=ax, label='CC')
ax.set_ylabel('CC')
ax = ax.twinx()
ax.set_ylabel('OA')
reference_counts.loc[2000:2025]['OA'].plot(ax=ax, label='OA', ls=':')
fig.legend(loc='upper center')
fig.tight_layout()

In [ ]:
def label_group_counts(
    df: pd.DataFrame,
    group: Group | AggTopic | AggAggTopic,
    geography_filter: Literal['affiliation', 'location', 'region_affiliation', 'region_location'] | None = None,
    count_primary_class: bool = False,
    threshold: float = 0.5,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    table_index = pd.RangeIndex(start=df['Publication year'].min(), stop=df['Publication year'].max() + 1, step=1, name='Publication year')

    columns = [label.column for label in group.labels if label.column in df.columns]
    mask = (df[columns].notna() & (df[columns] > threshold)).any(axis=1)
    primary = df[mask][columns].fillna(0).idxmax(axis=1)

    if group.collection == Collection.IMPACTS:
        mask &= df['incl_impacts']
    if geography_filter == 'location':
        mask &= df['incl_location']
    if geography_filter == 'affiliation':
        mask &= df['incl_affiliation']
    if geography_filter == 'region_location':
        mask &= df['incl_region_location']
    if geography_filter == 'region_affiliation':
        mask &= df['incl_region_affiliation']

    data = {}
    totals = {}
    for label in group.labels:
        if label.column not in df.columns:
            continue

        extra_mask = (primary == label.column) if count_primary_class else df[label.column] > threshold

        totals[label.name] = df[mask & extra_mask]['item_id'].nunique()
        data[label.name] = df[mask & extra_mask].groupby('Publication year')['item_id'].nunique()

    return (
        pd.DataFrame(data, index=table_index).fillna(0).astype(int),
        pd.Series(totals).fillna(0).astype(int),
        pd.Series(df[mask].groupby('Publication year')['item_id'].nunique(), index=table_index, name='Total').fillna(0).astype(int),
    )


# General counts

In [ ]:
print(df.shape)
print(df_locations.shape)
print(df_affiliations.shape)

In [ ]:
for key in ['incl_major', 'incl_impacts', 'incl_location', 'incl_affiliation']:
    print(key, df[key].sum())

# Annual counts

In [ ]:
pd.options.display.max_columns = 650
pd.options.display.max_rows = 40

def barplot(fname: str|None=None, twin: tuple[str,str,str] | None = None):
    GR =  1.618  # golden ratio
    fig_height = 5
    
    #['#fc8d62','#bb22b3','#8da0cb']
    deutera_colours =  ['#666E95', '#FFC13D', '#2F86E5', '#ACA6C3', '#483E3C','#E8C6BA', '#867065', '#005186', '#B0860F', '#FFD9B0']
    prota_colours = ['#7D7C01', '#59709E', '#4C5631', '#3F59E8', '#BDBB64','#35A9E0', '#E8E602', '#C1C1C7', '#0E1079']
    trita_colours = ['#5F727A', '#FFB7C2', '#01919A', '#A8A8B4', '#BDE6F4','#4A3F45', '#B77B87', '#00585C', '#932929']
    colours = deutera_colours
    
    fig, ax = plt.subplots(dpi=150, figsize=(fig_height * GR, fig_height))
    
    df_counts[('Global', 'Absolute')].drop(columns=['Total']).plot.bar(stacked=True, ax=ax, legend=False, color=colours)
    ax.set_ylabel('Number of publications')
    ax.set_xlabel('Publication year')
    ax.set_title(f'Estimated number of papers published on the nexus of climate and health')
    ax.set_xticks(np.arange(0, np.sum(ax.get_xlim()), 5))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

    if twin is not None:
        ax2 = ax.twinx()
        ax2.plot(np.arange(len(df_counts.index)), df_counts[('Global', 'Annual share of CC', 'Total')], ls=':', c='black', lw=1)
        ax2.set_ylabel('Share of climate change literature')
        ax2.set_ylim(0, 5)
    
    fig.tight_layout()

    if fname is not None:
        fig.savefig(OUTPUT_DIR / f'{fname}.png')
        fig.savefig(OUTPUT_DIR / f'{fname}.svg')
        fig.savefig(OUTPUT_DIR / f'{fname}.pdf')

    return fig, ax



def lineplot(df_counts_: pd.DataFrame, fname: str|None = None, num_subrows: int = 2):
    num_subs = len(df_counts_.T.groupby(level=0))
    num_stats = len(df_counts_.T.groupby(level=1)) - 1
    num_cols = math.ceil((num_stats+1) / num_subrows)

    fig = plt.figure(layout='constrained', figsize=(num_cols * 3, num_subs * num_subrows * 2), dpi=150)
    subfig = fig.subfigures(1, 1)
    for grouping, data in df_counts_.T.groupby(level=0):
        subfig.suptitle(grouping, fontweight='bold')
        axes = subfig.subplots(num_subrows, num_cols)
        if num_subrows == 1:
            axes = [axes]

        for ai, (subgroup, subdata) in enumerate(data.groupby(level=1)):
            j = ai % num_cols
            i = ai // num_cols
            for col, coldata in subdata.iterrows():
                axes[i][j].plot(coldata.index, coldata.fillna(np.nan).values, label=col[-1], ls='-' if col[-1] == 'Total' else '--')
            axes[i][j].set_title(subgroup)
            axes[i][j].legend(fontsize=7)

    #fig.tight_layout()

    if fname is not None:
        fig.savefig(OUTPUT_DIR / f'{fname}.png')
        fig.savefig(OUTPUT_DIR / f'{fname}.svg')
        fig.savefig(OUTPUT_DIR / f'{fname}.pdf')
    return fig



def annual_counts_info(count_primary_class: bool, label_group: str, mask: pd.Series):
    annual_counts, totals, annual_totals = label_group_counts(
        df[mask], 
        group=LABELS[label_group],
        geography_filter=None,
        count_primary_class=count_primary_class,
    )
    
    joined = annual_counts.join(annual_totals)
    cols = joined.columns
    share = (joined.T / joined['Total'] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual proportion'], cols]), axis=1)
    sharecc = (joined.T / reference_counts.loc[2006:2025]['CC'] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual share of CC'], cols]), axis=1)
    shareccpct = (sharecc['Annual share of CC'].pct_change(axis=0)*100).set_axis(pd.MultiIndex.from_product([['PCT of CC share'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    #shareoa = (joined.T / reference_counts.loc[2006:2025]['OA'] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual share of OA'], cols]), axis=1)
    cumsum = joined.cumsum().set_axis(pd.MultiIndex.from_product([['CumSum'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    sharecagr10 = ((((sharecc.pct_change(10) + 1) ** (1/10)) - 1) * 100).set_axis(pd.MultiIndex.from_product([['10yr CAGR CC'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    pct = (joined.pct_change(axis=0)*100).set_axis(pd.MultiIndex.from_product([['Annual growth in %'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    cagr5 = ((((joined.pct_change(5) + 1) ** (1/5)) - 1) * 100).set_axis(pd.MultiIndex.from_product([['5yr CAGR'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    cagr10 = ((((joined.pct_change(10) + 1) ** (1/10)) - 1) * 100).set_axis(pd.MultiIndex.from_product([['10yr CAGR'], cols]), axis=1).replace({np.inf:pd.NA, np.nan: pd.NA})
    df_global = joined.set_axis(pd.MultiIndex.from_product([['Absolute'], cols]), axis=1).join(cumsum).join(pct).join(cagr5).join(cagr10).join(share).join(sharecc).join(sharecagr10).join(shareccpct)

    df_global_total = pd.DataFrame({
        'Count (2006–2025)': df_global.loc[2006:2025]['Absolute'].sum(axis=0),
        'Count (2006–2015)': df_global.loc[2006:2015]['Absolute'].sum(axis=0),
        'Count (2016–2025)': df_global.loc[2016:2025]['Absolute'].sum(axis=0),
        'Share of total (2006–2025)': df_global.loc[2006:2025]['Absolute'].sum(axis=0) / df_global.loc[2006:2025][('Absolute', 'Total')].sum() * 100,
        'Share of total (2006–2015)': df_global.loc[2006:2015]['Absolute'].sum(axis=0) / df_global.loc[2006:2015][('Absolute', 'Total')].sum() * 100,
        'Share of total (2016–2025)': df_global.loc[2016:2025]['Absolute'].sum(axis=0) / df_global.loc[2016:2025][('Absolute', 'Total')].sum() * 100,
        'Share of CC (2006–2025)': df_global.loc[2006:2025]['Absolute'].sum(axis=0) / reference_counts.loc[2006:2025]['CC'].sum() * 100,
        'Share of CC (2006–2015)': df_global.loc[2006:2015]['Absolute'].sum(axis=0) / reference_counts.loc[2006:2015]['CC'].sum() * 100,
        'Share of CC (2016–2025)': df_global.loc[2016:2025]['Absolute'].sum(axis=0) / reference_counts.loc[2016:2025]['CC'].sum() * 100,
        'Share of OA (2006–2025)': df_global.loc[2006:2025]['Absolute'].sum(axis=0) / reference_counts.loc[2006:2025]['OA'].sum() * 100,
        'Share of OA (2006–2015)': df_global.loc[2006:2015]['Absolute'].sum(axis=0) / reference_counts.loc[2006:2015]['OA'].sum() * 100,
        'Share of OA (2016–2025)': df_global.loc[2016:2025]['Absolute'].sum(axis=0) / reference_counts.loc[2016:2025]['OA'].sum() * 100,
        'Growth 2006–2015': (df_global.loc[2015] / df_global.loc[2006] * 100)['Absolute'],
        'Growth 2016–2025': (df_global.loc[2025] / df_global.loc[2016] * 100)['Absolute'],
        'Growth (2006–2015) -> (2016–2025)': (df_global.loc[2016:2025]['Absolute'].sum(axis=0) / df_global.loc[2006:2015]['Absolute'].sum(axis=0) * 100),
    }).T

    display(df_global_total)
    
    total_global = df_global[('Absolute', 'Total')]
    
    df_global = df_global.set_axis(pd.MultiIndex.from_tuples([('Global', *col) for col in df_global.columns], names=['Selection','Metric','Category']), axis=1)

    display(Markdown('**Annual metrics**'))
    table = df_global
    display(table.style.background_gradient(cmap='Blues', axis=0))

    return table

## Primary class

In [ ]:
df_counts = annual_counts_info(count_primary_class=True, label_group='cat', mask=df['incl_major'])

fig, ax = barplot(fname='major_category', twin=('Global', 'Annual share of CC', 'Total'))
display(fig)
fig.clf()

fig = lineplot(df_counts, fname='annual_stats_primary')
display(fig)
fig.clf()

In [ ]:
idx = pd.IndexSlice
fig = lineplot(df_counts.loc[:, idx['Global', ['Absolute', 'Annual proportion', 'Annual share of CC', 'Annual growth in %'], :]], fname='annual_stats_primary_selection', num_subrows=1)

## Multi-label

In [ ]:
df_counts = annual_counts_info(count_primary_class=False, label_group='cat', mask=df['incl_major'])

fig = lineplot(df_counts, fname='annual_stats_multi')
display(fig)
fig.clf()

In [ ]:
idx = pd.IndexSlice
fig = lineplot(df_counts.loc[:, idx['Global', ['Absolute', 'Annual proportion', 'Annual share of CC', 'Annual growth in %'], :]], fname='annual_stats_multi_selection', num_subrows=1)

# Geographic distribution

In [ ]:
def plot(mask = None, bins: list[int] | None = None, show_tables: bool = False, show_distributions: bool = False):
    if mask is None:
        mask = np.ones(len(df_locations), dtype=bool)
    
    cnt_adm0 = df_locations[mask].groupby('iso3')['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'})
    cnt_latlon = df_locations[mask & (df_locations['feature_code'] != 'PCLI')].groupby(['LAT', 'LON'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'})
    
    cnt_adm0_yr = df_locations[mask].groupby(['iso3','Publication year'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'}).pivot(index='Publication year', columns='iso3', values='Count').fillna(0).astype(int)
    cnt_latlon_yr = df_locations[mask & (df_locations['feature_code'] != 'PCLI')].groupby(['LAT', 'LON', 'Publication year'])['item_id'].nunique().reset_index().rename(columns={'item_id': 'Count'}).pivot(index='Publication year', columns=['LAT','LON'], values='Count').fillna(0).astype(int)

    if show_tables:
        display(Markdown('### Number of publications per country'))
        display(cnt_adm0.transpose())
        display(cnt_latlon.transpose())
    
        display(Markdown('### Number of publications per country per year'))
        display(cnt_adm0_yr.transpose())
        display(cnt_latlon_yr.transpose())

    if show_distributions:
        display(Markdown('### Distribution of sub-national locations'))
        display(Markdown('\n'.join([
            f'* Percentile {p}: {int(np.percentile(cnt_latlon['Count'], p))}'
            for p in range(90, 101, 1)
        ])))
        fig, axes = plt.subplots(1,4, figsize=(15,3))
        cnt_latlon['Count'].hist(bins=100, ax=axes[0])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 99)]['Count'].hist(bins=50, ax=axes[1])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 98)]['Count'].hist(bins=20, ax=axes[2])
        cnt_latlon[cnt_latlon['Count'] < np.percentile(cnt_latlon['Count'], 95)]['Count'].hist(bins=10, ax=axes[3])
        plt.show(fig)
    
    fig = plt.figure(dpi=150)#, figsize=(10, 6.5))
    ax = plt.subplot(projection=AXIS_PROJECTION())
    ax.set_title('Locations of studies on the nexus of climate and health', loc='center', fontsize=10)
    ax.set_extent([-180, 180, -90, 90])
    ax.gridlines()
    
    cmap = mpl.colormaps['BuPu']
    norm = mpl.colors.Normalize(vmin=0, vmax=cnt_adm0['Count'].max())
    
    cnt_adm0.set_index('iso3', inplace=True)
    for _, row in shapes.iterrows():
        iso = row['ISO_A3']
        ax.add_geometries(
            [row['geometry']],
            fc=cmap(norm(cnt_adm0.loc[iso, 'Count'])) if iso in cnt_adm0.index else 'None', 
            crs=PROJECTION(), 
            ec='black', 
            lw=0.2,
        )
    
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=cmap), 
        ax=ax, orientation='horizontal',
        fraction=0.046, pad=0.04
    )
    cbar.set_label('Number of publications mentioning countries')

    
    if bins is not None:
        # Draw bubbles
        norm = mpl.colors.Normalize(vmin=0, vmax=np.percentile(cnt_latlon['Count'], 98))
        ax.scatter(
            cnt_latlon['LON'], cnt_latlon['LAT'], 
            transform=PROJECTION(), 
            s=norm(cnt_latlon['Count']) * 100,
            alpha=0.7,
            color='#ef8a62',
            zorder=10
        )
        
        # Prepare legend for bubbles
        for bin_i in bins: 
            ax.scatter(
                [],[], 
                transform=PROJECTION(), 
                s=norm(bin_i) * 100,
                label=bin_i,
                alpha=0.7,
                color='#ef8a62',
            )

        leg = ax.legend(title='Number of publications\nmentioning sub-national\nlocations', fontsize=8)
        leg.get_title().set_fontsize('8')
    
    fig.tight_layout()
    
    return fig

## Statistics 1990–2025
These counts include the number of records included in the global indicator that mention a location in Latin America. The analysis has four resolutions: 
* Number of publications per country,
* Number of publications per country and year,
* Number of publications per grid cell and year (2-digits,  [~1km resolution](https://support.oxts.com/hc/en-us/articles/115002885125-Level-of-Resolution-of-Longitude-and-Latitude-Measurements)), and
* Number of publications per grid cell (2-digits,  [~1km resolution](https://support.oxts.com/hc/en-us/articles/115002885125-Level-of-Resolution-of-Longitude-and-Latitude-Measurements)).

The first and last aggregations are plotted.    
For comparison, the distribution and percentiles of counts per grid cell are shown as well.   
The grid cell aggregation is excludion location with 'PCLI' admin codes [(freely associated state)](https://www.geonames.org/export/codes.html)

In [ ]:
fig = plot(show_tables=False, show_distributions=False)

fig.savefig(OUTPUT_DIR / 'geographic_map.png')
fig.savefig(OUTPUT_DIR / 'geographic_map.svg')
fig.savefig(OUTPUT_DIR / 'geographic_map.pdf')

In [ ]:
fig = plot(bins=[1, 10, 50, 100], show_tables=False, show_distributions=True)
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.png')
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.svg')
fig.savefig(OUTPUT_DIR / 'geographic_map_bubbles.pdf')

## Statistics 2025

In [ ]:
fig = plot(show_tables=False, show_distributions=False, mask=df_locations['Publication year']==2025)

fig.savefig(OUTPUT_DIR / 'geographic_map_2025.png')
fig.savefig(OUTPUT_DIR / 'geographic_map_2025.svg')
fig.savefig(OUTPUT_DIR / 'geographic_map_2025.pdf')

# Author affiliations vs study locations

In [ ]:
for grouping in [
    'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    'Group (HDI 2026)', 
    'Region (WorldBank 2026)', 
    'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
    #'Name (Lancet 2026)'
]:
    display(Markdown(f'## {grouping}'))
    groups = df_countries[df_countries[grouping].notna()][grouping].unique()
    
    subgroups = {
        name: [
            (f'{name} ({len(index):,})', index)
            for name, index in filters
        ]
        
        for name, filters in {
            'Study location': [
                ('None found', df[~df.index.isin(df_locations.index)].index.unique()),
                ('Ungrouped', df_locations[df_locations[grouping].isna()].index.unique()),
            ] + [
                (group, df_locations[df_locations[grouping]==group].index.unique())
                for group in groups
            ],
            'Author affiliation': [
                ('Not available', df[~df.index.isin(df_affiliations.index)].index.unique()),
                ('Ungrouped', df_affiliations[df_affiliations[grouping].isna()].index.unique()),
            ] + [
                (group, df_affiliations[df_affiliations[grouping]==group].index.unique())
                for group in groups
            ],
        }.items()
    }
    
    data = []
    for name1, index1 in subgroups['Study location']:
        for name2, index2 in subgroups['Author affiliation']:
            data.append({
                'Weight': len(index1.intersection(index2)),
                'Study location': name1,
                'Author affiliation': name2,
            })
    sdf = pd.DataFrame(data)
    fig = plt.figure(figsize=(7.4, 7), dpi=150)
    sankey(sdf, cmap=plt.get_cmap('viridis'), labels_size=8, titles_size=10)
    plt.tight_layout()
    display(fig)

    
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.png')
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.svg')
    fig.savefig(OUTPUT_DIR / f'sankey_affil_location_{grouping}.pdf')
    
    fig.clf()
    fig.clear()

# Regions

In [ ]:
def produce_regional_counts(grouping: str, df_geo: pd.DataFrame, prefix: str):
    regions = [region for region in df_geo[grouping].unique() if type(region) is str]
    md = f'### {grouping}\n * Unique: {regions}'
    
    if grouping in ORDERING:
        regions = ORDERING[grouping]
        md += f'\n * Ordered: {regions}'
        if len(set(regions) ^ set(ORDERING[grouping])) > 0:
            md+=f'\n * **Set mismatch!!**'
    display(Markdown(md))
    
    mask = df['incl_major'] & df['incl_location'] & df['incl_py']
    
    buffer = []
    buffer_totals = []
    for region in regions:
        if type(region) is not str:
            continue
        logger.info(f'Counting for "{region}" in {grouping} (mask: {(mask & df.index.isin(df_geo[df_geo[grouping] == region]['item_id'])).sum():,}/{mask.sum():,}/{mask.shape[0]:,})')
        annual_counts, totals, annual_totals = label_group_counts(
            df[mask & df.index.isin(df_geo[df_geo[grouping] == region]['item_id'])], 
            group=LABELS['cat'],
            geography_filter='location',
            count_primary_class=True,
        )
        annual_counts['Total'] = annual_totals
        annual_counts.columns = pd.MultiIndex.from_product([[region], annual_counts.columns])
        totals['Total'] = annual_totals.sum()
        totals.index = pd.MultiIndex.from_product([[region], totals.index])
        buffer_totals.append(totals)
        buffer.append(annual_counts)
    regional_counts = pd.concat(buffer, axis=1)
    regional_totals = pd.concat(buffer_totals)
    annual_total = df[mask].groupby('Publication year')['item_id'].nunique()

    annual_group_population = df_population.merge(df_countries, left_on='iso3', right_on='iso3', how='left').groupby([grouping, 'year'])['Population'].sum().reset_index().pivot(index='year', values='Population', columns=grouping)
    annual_group_population.loc[2025] = annual_group_population.loc[2024]  # No data for 2025 yet
    
    prc_pop = regional_counts / annual_group_population.rename_axis(index='Publication year', columns=None) * 100000
    prc_cagr = ((((regional_counts.cumsum().pct_change(5) + 1) ** (1/5)) - 1) * 100).iloc[5:].replace({np.inf:pd.NA, np.nan: pd.NA})
    prc_pct = regional_counts.cumsum().pct_change().replace({np.inf:pd.NA, np.nan: pd.NA}) * 100
    prc_per = ((regional_counts.T / regional_counts.T.xs('Total', level=1)).T*100).replace({np.inf:pd.NA, np.nan: pd.NA})
    prc_sha = (regional_counts.T / annual_total*100).replace({np.inf:pd.NA, np.nan: pd.NA}).T
    
    mask = df['incl_major'] & df['incl_location'] & df.index.isin(df_geo[df_geo[grouping].isin(regions)]['item_id']) & df['incl_py']

    display(Markdown(f'#### Totals\nNumber of unique records: {df[mask].index.nunique():,}'))
    display(pd.DataFrame({
        'Total (abs)': regional_totals.astype(int),
        'Total (share per group)':regional_totals/ regional_totals.xs('Total', level=1) * 100,
        'Total (share of total w/ loc, 2006–2025)': regional_totals / df[mask].index.nunique() * 100,
        'Total (share of total w/ loc, 2006–2015)': regional_counts.loc[2006:2015].sum() / df[mask & (df['Publication year']>=2006) & (df['Publication year']<= 2015)].index.nunique() * 100,
        'Total (share of total w/ loc, 2016–2025)': regional_counts.loc[2016:2025].sum() / df[mask & (df['Publication year']>=2016) & (df['Publication year']<= 2025)].index.nunique() * 100,
        'Total (share of total, 2006–2025)': regional_totals / df[df['incl_major']].index.nunique() * 100,
        'Total (share of total, 2006–2015)': regional_counts.loc[2006:2015].sum() / df[df['incl_major'] & (df['Publication year']>=2006) & (df['Publication year']<= 2015)].index.nunique() * 100,
        'Total (share of total, 2016–2025)': regional_counts.loc[2016:2025].sum() / df[df['incl_major'] & (df['Publication year']>=2016) & (df['Publication year']<= 2025)].index.nunique() * 100,
        'Total (per 100k citizens, 2006–2025)': prc_pop.loc[2006:2025].mean(axis=0),
        'Total (per 100k citizens, 2006–2015)': prc_pop.loc[2006:2015].mean(axis=0),
        'Total (per 100k citizens, 2016–2025)': prc_pop.loc[2016:2025].mean(axis=0),
    }).T)

    fig, axes = plt.subplots(1, 5, figsize=(25, 2.5), dpi=150)

    for ai, (sub, title, lim) in enumerate([
        (regional_counts, 'Annual counts', None),
        (prc_sha, 'Annual shares', None),
        (prc_pct, 'Annual growth', (0, 80)),
        (prc_cagr, 'CAGR @ 5yrs', None),
        (prc_pop, 'Records per 1M citizens', None),
    ]):
        tab = sub.T.xs('Total', level=1).T.fillna(np.nan)
        for col in tab.columns:
            axes[ai].plot(tab.index, tab[col], label=col, lw=1)
            axes[ai].set_title(title)
        if lim is not None:
            axes[ai].set_ylim(*lim)
    axes[-1].legend(bbox_to_anchor=(1.1, 1.05))

    display(fig)
    
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.png')
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.svg')
    fig.savefig(OUTPUT_DIR / f'regional_counts_{prefix}_{grouping}.pdf')
    
    fig.clf()
    fig.clear()

    display(Markdown('#### Annual counts'))
    display(regional_counts.astype('Int64'))

    display(Markdown('#### Annual shares per group'))
    display(prc_per)
    
    display(Markdown('#### Annual shares of all'))
    display(prc_sha)

    display(Markdown('#### Annual growth'))
    display(prc_pct)
    
    display(Markdown('#### CAGR @ 5yrs'))
    display(prc_cagr)

    display(Markdown('#### Annual counts by population\nNumbers are normalised to records per 100,000 citizens'))
    display(prc_pop)

## Regions by location

In [ ]:
for grp in [
    'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    'Group (HDI 2026)', 
    'Group (HDI-2 2026)', 
    #'Region (WorldBank 2026)', 
    'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
]:
    produce_regional_counts(grp, df_geo=df_locations, prefix='locations')

## Regional counts by affiliation

In [ ]:
for grp in [
    'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    'Group (HDI 2026)', 
    'Group (HDI-2 2026)', 
    #'Region (WorldBank 2026)', 
    'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
]:
    produce_regional_counts(grp, df_geo=df_affiliations, prefix='affiliations')

## Affiliation stats
### Number of authors per record

In [ ]:
tab = df_affiliations.reset_index(drop=True).groupby(['Publication year', 'item_id'])['author_id'].nunique().groupby('Publication year').agg(['mean', 'std'])
display(tab.T)
display(df_affiliations.reset_index(drop=True).groupby(['Publication year', 'item_id'])['author_id'].nunique().agg(['mean', 'std']).T)

fig, ax = plt.subplots(dpi=150, figsize=(15,3))
ax.errorbar(tab.index, tab['mean'], tab['std'], linestyle='None', marker='^')
tab['mean'].plot(ax=ax)
ax.set_title('Average number of authors (with std) per record')

fig.savefig(OUTPUT_DIR / 'affiliation_means.png')
fig.savefig(OUTPUT_DIR / 'affiliation_means.svg')
fig.savefig(OUTPUT_DIR / 'affiliation_means.pdf')

# Climate impacts

In [ ]:
df_imp = pd.DataFrame({
    'Relevant records': df[df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant with location': df[df['incl_location'] & df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant records and category impacts': df[df['incl_major'] & (df['cat|2']>0.5)].groupby('Publication year')['item_id'].nunique(),
    'Relevant on impacts': df[df['incl_impacts'] & df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    'Relevant on impacts with location': df[df['incl_location'] & df['incl_major'] & df['incl_impacts']].groupby('Publication year')['item_id'].nunique(),
    'Records on impacts with location in attributable cell':  df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
    'Records on impacts with location in non-attributable cell': df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & ~df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
})
display(Markdown('## Totals'))
display(pd.DataFrame({
    '2006–2015': df_imp.loc[2006:2015].sum(),
    '2016–2025': df_imp.loc[2016:2025].sum(),
    '2006–2025': df_imp.loc[2006:2025].sum(),
}).T)

display(Markdown('## Annual counts'))
display(pd.concat([df_imp, df_imp.sum().rename('Total').to_frame().T]))

display(Markdown('## Cummulative counts'))
display(df_imp.cumsum())

display(Markdown('## Annual growth'))
display(df_imp.cumsum().pct_change()*100)

## Label counts

In [ ]:
pd.options.display.max_columns = 650
pd.options.display.max_rows = 40

incl_attr = df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable']

for key in [
    'driver', 
    'event', 
    'health', 
    'expose',
    'attr',
    'sector', 
    'keywords',
   #'topic-agg',
    'topic-agg-agg',
]:
    group = LABELS[key]

    display(Markdown(f'### {group.name}'))
    
    annual_counts, totals, annual_totals = label_group_counts(
        #df[df.index.isin(incl_ids)], 
        df[incl_attr],
        group=group,
        geography_filter=None,
        count_primary_class=False,
    )

    joined = annual_counts.join(annual_totals)
    cols= joined.columns
    share = (joined.T / joined['Total'] * 100).T.set_axis(pd.MultiIndex.from_product([['Annual proportion'], cols]), axis=1)
    pct = (joined.cumsum().pct_change(axis=0)*100).set_axis(pd.MultiIndex.from_product([['Annual growth in %'], cols]), axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)#.astype('Float32')
    cumsum = joined.cumsum().set_axis(pd.MultiIndex.from_product([['CumSum'], cols]), axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)
    stats = joined.set_axis(pd.MultiIndex.from_product([['Absolute'], cols]), axis=1).join(pct).join(share).join(cumsum)
    display(pd.DataFrame({
        'Count (2006–2015)': stats.loc[2006:2015]['Absolute'].sum(axis=0),
        'Count (2016–2025)': stats.loc[2016:2025]['Absolute'].sum(axis=0),
        'Count (2006–2025)': stats.loc[2006:2025]['Absolute'].sum(axis=0),
        'Share of total (2006–2015)': stats.loc[2006:2015]['Absolute'].sum(axis=0) / stats.loc[2006:2015][('Absolute', 'Total')].sum() * 100,
        'Share of total (2016–2025)': stats.loc[2016:2025]['Absolute'].sum(axis=0) / stats.loc[2016:2025][('Absolute', 'Total')].sum() * 100,
        'Share of total (2006–2025)': stats.loc[2006:2025]['Absolute'].sum(axis=0) / stats.loc[2006:2025][('Absolute', 'Total')].sum() * 100,
        'Growth 2006–2015': (stats.loc[2015] / stats.loc[2006] * 100)['Absolute'].replace([np.inf, -np.inf], np.nan),
        'Growth 2016–2025': (stats.loc[2025] / stats.loc[2016] * 100)['Absolute'].replace([np.inf, -np.inf], np.nan),
        'Growth (2006–2015) -> (2016–2025)': (stats.loc[2016:2025]['Absolute'].sum(axis=0) / stats.loc[2006:2015]['Absolute'].sum(axis=0) * 100).replace([np.inf, -np.inf], np.nan),
        
    }).T.style.background_gradient(cmap='Blues', axis=1))
    display(stats.style.background_gradient(cmap='Blues', axis=0))

## Health impacts per region

In [ ]:
for label in LABELS['health'].labels:
    print(f"('{label.name}', ['{label.column}']),")

In [ ]:
data = df[df['incl_major'] & df['incl_impacts'] & df['incl_location'] & df['grid_attributable']].drop(columns=['abstract']).join(df_locations.set_index('item_id', drop=False), rsuffix='_')

seen_cols = set()
counts = {}
grouping = 'Group (HDI 2026)'
for name, columns in [
    ('Infectious diseases', ['health|4']),
    ('Mortality and morbidity', ['health|6', 'health|3']), # 'Mortality and morbidity (general)' + 'Direct injury and death'
    ('Respiratory disease', ['health|12']),
    ('Food and water safety and security', ['health|5', 'health|0']), # 'Water safety and security' + 'Food safety and security'
    ('Mental health and sentiment', ['health|1']),
    ('Cardiovascular disease', ['health|2']),
    ('Infectious diseases', ['health|4']),
    ('Maternal, reproductive and infant health', ['health|8']),
    ('Health system capacity', ['health|7']),
    
    #('Renal system', ['health|9']),
    #('Metabolic Disorders', ['health|10']),
    #('Other / unspecified health impacts', ['health|11']),
]:
    seen_cols |= set(columns)
    counts[name] = data[(data[columns]>0.5).any(axis=1)].groupby(grouping)['item_id'].nunique()
    counts[name].loc['Total'] = data[(data[columns]>0.5).any(axis=1)]['item_id'].nunique()
counts['Other'] = data[(data[list(seen_cols)]<=0.5).all(axis=1)].groupby(grouping)['item_id'].nunique()
counts['Other'].loc['Total'] = data[(data[list(seen_cols)]<=0.5).all(axis=1)]['item_id'].nunique()
counts['Total'] = data.groupby(grouping)['item_id'].nunique()
counts['Total'].loc['Total'] = data['item_id'].nunique()

df_cnt = pd.DataFrame(counts).T[ORDERING[grouping]+['Total']]

display(df_cnt)

In [ ]:
def get_contrast_color(value, cmap, norm):
    """Returns 'white' or 'black' based on the background color intensity."""
    rgba = cmap(norm(value))
    # Calculate brightness (Luma) using standard coefficients
    brightness = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    return "white" if brightness < 0.5 else "black"

# 1. Prepare the data (removing totals from the main heatmap)
# Assuming df_cnt is your DataFrame from the screenshot
data_only = df_cnt.drop('Total', axis=0).drop('Total', axis=1)
row_totals = df_cnt.loc[data_only.index, 'Total']
col_totals = df_cnt.loc['Total', data_only.columns]
norm_data = data_only / df_cnt.loc['Total', 'Total']

# 2. Setup the figure and GridSpec
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[4, 1], wspace=0.05, hspace=0.05)

ax_main = plt.subplot(gs[0, 0])
ax_row_bar = plt.subplot(gs[0, 1], sharey=ax_main)
ax_col_bar = plt.subplot(gs[1, 0], sharex=ax_main)

cmap = plt.cm.Greens
# Using a LogNorm can sometimes make light values too pale; 
# if legibility is an issue, consider Normalize() instead.
norm = plt.Normalize(vmin=norm_data.values.min(), vmax=norm_data.values.max())

# --- 3. Main Heatmap ---
im = ax_main.imshow(norm_data, aspect='auto', cmap=cmap, norm=norm)

ax_main.set_yticks(range(len(data_only.index)))
ax_main.set_yticklabels(data_only.index)
ax_main.set_xticks(range(len(data_only.columns)))
ax_main.set_xticklabels(data_only.columns)#, rotation=0, ha="right")
ax_main.xaxis.set_ticks_position('top')
ax_main.xaxis.set_label_position('top')

# Add text annotations to heatmap
for i in range(len(data_only.index)):
    for j in range(len(data_only.columns)):
        text_color = get_contrast_color(norm_data.iloc[i, j], cmap, norm)
        ax_main.text(j, i, 
                     f'{norm_data.iloc[i, j]:.0%}\n({data_only.iloc[i, j]:,})',
                     ha="center", va="center", color=text_color)# color="w")

# --- 4. Right Bar Chart (Row Totals) ---
ax_row_bar.barh(range(len(row_totals)), row_totals, color='#2ca02c', alpha=0.5)
ax_row_bar.set_axis_off()
# Add labels to bars
for i, v in enumerate(row_totals):
    ax_row_bar.text(v if v < 4000 else 0,
                    i, f'    {v / df_cnt.loc['Total', 'Total']:.0%}\n  {v:,}', va='center', fontsize=9)

# --- 5. Bottom Bar Chart (Column Totals) ---
ax_col_bar.bar(range(len(col_totals)), col_totals, bottom=col_totals.max() - col_totals, color='#2ca02c', alpha=0.5)
ax_col_bar.set_axis_off()
# Add labels to bars
for i, v in enumerate(col_totals):
    ax_col_bar.text(i, 
                    col_totals.max() - v + 500 if v > 3000 else col_totals.max() - 8000, 
                    f' {v / df_cnt.loc['Total', 'Total']:.0%}\n{v:,}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Publications per Health Impact and HDI Group', y=0.95, x=0.45)

fig.savefig(OUTPUT_DIR / 'health_hdi.png')
fig.savefig(OUTPUT_DIR / 'health_hdi.svg')
fig.savefig(OUTPUT_DIR / 'health_hdi.pdf')

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(df_cnt)
ax.set_xticks(range(df_cnt.shape[1]), labels=df_cnt.columns, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(df_cnt.shape[0]), labels=df_cnt.index)

for i, (ii, row) in enumerate(df_cnt.iterrows()):
    for j, (jj, cell) in enumerate(row.items()):
        text = ax.text(j, i, f'{cell/1000:.1f}', ha="center", va="center", color="w")

ax.set_title('Number of publication per health impact and HDI group (x 1,000)')
fig.tight_layout()

## Attributions on geographic map

In [ ]:
lim_percentile = (0, np.percentile(grid_counts[grid_counts['studies_per_capita'].notna()]['studies_per_capita'], 0.99))
lim_median = (0, grid_counts[grid_counts['studies_per_capita'].notna()]['studies_per_capita'].median() * 50)
print(f'Upper 95th percentile limit for per capita: {lim_percentile}')
print(f'50x median limit for per capita: {lim_median}')

grid_counts.describe()

### Population / number of studies plots
Figure includes *all* impacts studies!
Overlay shows where we have attributable effects (`//` for OR, `xx` for AND)

In [ ]:
#mpl.rcParams['hatch.linewidth'] = 0.1  # previous pdf hatch linewidth
#mpl.rcParams['hatch.linewidth'] = 1.0  # previous svg hatch linewidth
mpl.rcParams['hatch.linewidth'] = 0.2

for attribution_layer in ['', '_with_attribution_land', '_with_attribution_all']:
    fig, axes = plt.subplots(1,3, dpi=150, figsize=(16, 5), subplot_kw={'projection': ccrs.EqualEarth()})
    for ax, (field, name, scaling, lim, norm) in zip(axes, [
        ('population', 'Number of citizens', 1, None, True),
        ('study_count', 'Number of publications', 1, None, True),
        ('studies_per_capita', 'Publications per 1,000,000 citizens', 1000000, lim_median, True),
    ]):
        grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values=field).astype('float32').replace({pd.NA: np.nan}).fillna(np.nan)
        if lim:
            grid = grid.clip(*lim)
        mesh = ax.pcolormesh(
            # LON, LAT, DATA
            grid.columns, grid.index, grid.values * scaling,
            cmap=mpl.colormaps.get_cmap('Greens'),
            norm=mpl.colors.LogNorm() if norm else None,
            transform=ccrs.PlateCarree(),
        )

        if attribution_layer != '':
            if attribution_layer == '_with_attribution_land':
                grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values='attribution')
            else: # _with_attribution_all
                grid = pd.pivot_table(grid_counts, index='LAT', columns='LON', values='attribution')

            patterns = ax.contourf(
                grid.columns, grid.index, grid.values,
                levels=[-0.5, 0.5, 1.5, 2.5], 
                # '': transparent, '.': dotted, '\\': lines
                hatches=['', '/////', 'xxxxxxx'],
                colors='none', # Makes the background of the hatch transparent
                transform=ccrs.PlateCarree(),
                corner_mask=False,
            )
        
        ax.set_extent([-180, 180, -90, 90])
        ax.coastlines(lw=0.2)
        
        fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05, label=name)
    
    fig.tight_layout()
    
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.png')
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.svg')
    fig.savefig(OUTPUT_DIR / f'global_distribution{attribution_layer}.pdf')

    display(fig)
    
    fig.clf()
    fig.clear()

### Per capita and number of study plots
Because we have two types of locations, countries (etc) and cities (etc), we need the figure above has two sources of the counts per grid cell.
This plot shows both sources individually.

In [ ]:
fig, axes = plt.subplots(2,2, dpi=150, figsize=(16,10), subplot_kw={'projection': ccrs.EqualEarth()})

for ax, field, name, scaling, lim, norm in [
    (axes[0][0], 'study_count_abstracted', 'Number of publications (country-level etc)', 1, None, True),
    (axes[0][1], 'studies_per_capita_abstracted', 'Publications per 1,000,000 citizens (country-level etc)', 1000000, None, True),
    (axes[1][0], 'study_count_direct', 'Number of publications (regional etc)', 1, None, True),
    (axes[1][1], 'studies_per_capita_direct', 'Publications per 1,000,000 citizens (regional etc)', 1000000, lim_median, True),
]:
    grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values=field).astype('float32').replace({pd.NA: np.nan}).fillna(np.nan)
    if lim:
        grid = grid.clip(*lim)
    mesh = ax.pcolormesh(
        # LON, LAT, DATA
        grid.columns, grid.index, grid.values * scaling,
        cmap=mpl.colormaps.get_cmap('Greens'),
        norm=mpl.colors.LogNorm() if norm else None,
        transform=ccrs.PlateCarree(),
    )
    ax.set_extent([-180, 180, -90, 90])
    
    ax.coastlines(lw=0.2)
    
    fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05, label=name)
fig.tight_layout()

fig.savefig(OUTPUT_DIR / 'global_distribution_split.png')
fig.savefig(OUTPUT_DIR / 'global_distribution_split.svg')
fig.savefig(OUTPUT_DIR / 'global_distribution_split.pdf')

### Attribution in one colormesh

In [ ]:
fig, ax = plt.subplots(1,1, dpi=150, subplot_kw={'projection': ccrs.EqualEarth(central_longitude=0)})

grid = pd.pivot_table(df_grid.mask(~df_grid['is_land']), index='LAT', columns='LON', values='attribution')

# F768A0
# AD017D
ax.coastlines(lw=0.2)
ax.pcolormesh(grid.columns, grid.index, grid.values * 0 + 1, cmap=mpl.colors.ListedColormap(["#000000"]), transform=ccrs.PlateCarree())
mesh = ax.pcolormesh(grid.columns, grid.index, grid.values, vmin=0, vmax=2, cmap=mpl.colors.ListedColormap(['#f1f1f1','#F768A0', '#AD017D']), transform=ccrs.PlateCarree())
bar = fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05)#, label='Attributable temperature trends')
bar.set_ticks(ticks=[0,1,2], labels=['No attributable trends', 'Temperature or\nprecipitation', 'Temperature and\nprecipitation'], fontsize=9)
ax.set_extent([-180,180, -90, 90])
fig.savefig(OUTPUT_DIR / 'attributable_onepanel.png')
fig.savefig(OUTPUT_DIR / 'attributable_onepanel.pdf')
fig.savefig(OUTPUT_DIR / 'attributable_onepanel.svg')

### Horizontal attribution panels

In [ ]:
tmp = df_grid.copy()
tmp['temp'] = 1
tmp.loc[tmp['grid_cooler'], 'temp'] = 0
tmp.loc[tmp['grid_warmer'], 'temp'] = 2

tmp['prec'] = 1
tmp.loc[tmp['grid_drier'], 'prec'] = 0
tmp.loc[tmp['grid_wetter'], 'prec'] = 2

fig, axes = plt.subplots(1,3, dpi=150, figsize=(16,7), subplot_kw={'projection': ccrs.EqualEarth(central_longitude=0)})

axes[0].set_title('a) Attributable trends in temperature')
axes[0].coastlines(lw=0.2)
grid = pd.pivot_table(tmp.mask(~tmp['is_land']), index='LAT', columns='LON', values='temp')
axes[0].pcolormesh(grid.columns, grid.index, grid.values * 0 + 1, cmap=mpl.colors.ListedColormap(["#000000"]), transform=ccrs.PlateCarree())
mesh = axes[0].pcolormesh(grid.columns, grid.index, grid.values, vmin=0, vmax=2, cmap=mpl.colors.ListedColormap(['#67a9cf', '#f1f1f1', '#ef8a62']), transform=ccrs.PlateCarree())
bar = fig.colorbar(mesh, ax=axes[0], location='bottom', shrink=0.75, pad=0.05)
bar.set_ticks(ticks=[0,1,2], labels=['Cooler', 'Not attributable', 'Warmer'], fontsize=9)


axes[1].set_title('b) Attributable trends in precipitation')
axes[1].coastlines(lw=0.2)
grid = pd.pivot_table(tmp.mask(~tmp['is_land']), index='LAT', columns='LON', values='prec')
axes[1].pcolormesh(grid.columns, grid.index, grid.values * 0 + 1, cmap=mpl.colors.ListedColormap(["#000000"]), transform=ccrs.PlateCarree())
mesh = axes[1].pcolormesh(grid.columns, grid.index, grid.values, vmin=0, vmax=2, cmap=mpl.colors.ListedColormap(['#ef8a62', '#f1f1f1', '#67a9cf']), transform=ccrs.PlateCarree())
bar = fig.colorbar(mesh, ax=axes[1], location='bottom', shrink=0.75, pad=0.05)
bar.set_ticks(ticks=[0,1,2], labels=['Drier', 'Not attributable', 'Wetter'], fontsize=9)


axes[2].set_title('c) Density of impact studies')
axes[2].coastlines(lw=0.2)
grid = pd.pivot_table(grid_counts.mask(~grid_counts['is_land']), index='LAT', columns='LON', values='studies_per_capita').astype('float32').replace({pd.NA: np.nan}).fillna(np.nan).clip(*lim_median) * 1000000
mesh = axes[2].pcolormesh(grid.columns, grid.index, grid.values, cmap=mpl.colormaps.get_cmap('Greens'), norm=mpl.colors.LogNorm(), transform=ccrs.PlateCarree())
fig.colorbar(mesh, ax=axes[2], location='bottom', shrink=0.75, pad=0.05, label='Publications per 1,000,000 citizens')

extent = [-180,180, -90, 90]
axes[0].set_extent(extent)
axes[1].set_extent(extent)
axes[2].set_extent(extent)

fig.tight_layout()
plt.show(fig)

fig.savefig(OUTPUT_DIR / 'attribution_horizontal.png')
fig.savefig(OUTPUT_DIR / 'attribution_horizontal.svg')
fig.savefig(OUTPUT_DIR / 'attribution_horizontal.pdf')

### Separated studies by climate driver

In [ ]:
print(df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable']].shape)
print(df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable'] & (df['driver|1']>0.5)].shape)
print(df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable'] & (df['driver|3']>0.5)].shape)
print(df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable'] & ((df['driver|1']>0.5) | (df['driver|3']>0.5))].shape)
print(df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable'] & (df['driver|1']>0.5) & (df['driver|3']>0.5)].shape)

In [ ]:
from climate_health_map.data.geographies import join_grid_shapes, join_shapes
reference_year=2024
resolution=2.5
location_codes_abstracted=LOCATION_FEATURE_CODES[0]
location_codes_direct=LOCATION_FEATURE_CODES[3] | LOCATION_FEATURE_CODES[2] | LOCATION_FEATURE_CODES[1]

study_mask = df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location']
mask_abstracted = df_locations['feature_code'].isin(location_codes_abstracted)
mask_direct = df_locations['feature_code'].isin(location_codes_direct)

# Match df_grid to shapes
points_grid = join_grid_shapes(df_grid, shapes=shapes, resolution=resolution)

In [ ]:
print(matched.shape)
display(matched.head())
print(points_grid.shape)
display(points_grid.head())

for d in LABELS['driver'].labels:
    print(d.name, d.column)

In [ ]:
for driver_mask, name, key, norm in [
    (df_locations['item_id'].isin(df[df['driver|1']>0.5].index), 'Studies mentioning changes in temperature', 'temp', False),
    (df_locations['item_id'].isin(df[df['driver|3']>0.5].index), 'Studies mentioning changes in precipitation', 'prec', False),
    (df_locations['item_id'].isin(df[df['driver|1']>0.5].index), 'Studies mentioning changes in temperature', 'temp_log', True),
    (df_locations['item_id'].isin(df[df['driver|3']>0.5].index), 'Studies mentioning changes in precipitation', 'prec_log', True),
]:
    # Match extracted locations to shapes
    matched = pd.concat([
        join_shapes(df_locations[study_mask & driver_mask & mask_direct]
                    .merge(df_grid.reset_index(drop=True), left_on=['LAT', 'LON'], right_on=['LAT', 'LON'])[['grid_id', 'item_id', 'location_id', 'lat', 'lon']],
                    shapes=shapes, keep_columns=['location_id', 'grid_id'])[['location_id', 'shape_id', 'item_id', 'grid_id']].assign(loc_direct=True),
        (
            df_locations[study_mask & driver_mask & mask_abstracted][['location_id', 'iso3', 'item_id']]
            .merge(shapes.reset_index(names='shape_id'), left_on='iso3', right_on='ISO_A3')[['location_id', 'shape_id', 'item_id']]
            .merge(points_grid, left_on='shape_id', right_on='shape_id')[['location_id', 'shape_id', 'item_id', 'grid_id']]
            .assign(loc_abstract=True)
        ),
    ])
    counts = matched.groupby('grid_id')['item_id'].nunique().to_frame().rename(columns={'item_id': 'count'}).merge(df_grid, left_on='grid_id', right_on='grid_id')
    fig, ax = plt.subplots(1,1, dpi=150, subplot_kw={'projection': ccrs.EqualEarth(central_longitude=0)})
    grid = pd.pivot_table(counts.mask(~counts['is_land']), index='LAT', columns='LON', values='count').astype('float32').replace({pd.NA: np.nan}).fillna(np.nan)
    mesh = ax.pcolormesh(
        # LON, LAT, DATA
        grid.columns, grid.index, grid.values,
        cmap=mpl.colormaps.get_cmap('Greens'),
        norm=mpl.colors.LogNorm() if norm else None,
        transform=ccrs.PlateCarree(),
    )
    ax.set_extent([-180, 180, -90, 90])
    
    ax.coastlines(lw=0.2)
    
    fig.colorbar(mesh, ax=ax, location='bottom', shrink=0.75, pad=0.05, label=name)
    
    fig.tight_layout()
    
    fig.savefig(OUTPUT_DIR / f'driver_{key}_global_distribution.png')
    fig.savefig(OUTPUT_DIR / f'driver_{key}_global_distribution.svg')
    fig.savefig(OUTPUT_DIR / f'driver_{key}_global_distribution.pdf')

    display(fig)
    
    fig.clf()
    fig.clear()


## Exposed regions

In [ ]:
totals = {
    ('Global', 'Total'): df.groupby('Publication year')['item_id'].nunique(),
    ('Global', 'C&H include'): df[df['incl_major']].groupby('Publication year')['item_id'].nunique(),
    ('Global', ' + impacts include'): df[df['incl_major'] & df['incl_impacts']].groupby('Publication year')['item_id'].nunique(),
    ('Global', ' + location'): df[df['incl_major'] & df['incl_impacts'] & df['incl_location']].groupby('Publication year')['item_id'].nunique(),
    ('Global', ' + attributable'): df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
    ('Global', ' / not attributable'): df[df['incl_location'] & df['incl_major'] & df['incl_impacts'] & ~df['grid_attributable']].groupby('Publication year')['item_id'].nunique(),
}

df_totals = pd.DataFrame(totals)

for grouping in [
    'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    'Group (HDI 2026)', 
    'Group (HDI-2 2026)', 
    #'Region (WorldBank 2026)', 
    'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
]:
    regions = [region for region in df_locations[grouping].unique() if type(region) is str]
    if grouping in ORDERING:
        if len(set(regions) ^ set(ORDERING[grouping])) > 0:
            display(f'Set mismatch:\n * {regions}\n * {ORDERING[grouping]}')
        
        regions = ORDERING[grouping]
        display(Markdown(f'### {grouping}\n{regions} (ordered)'))
    else:
        display(Markdown(f'### {grouping}\n{regions} (alphabetic order)'))
    
    buffer = copy(totals)
    for region in regions:
        grp_msk = df.index.isin(df_locations[df_locations[grouping] == region].index)
        buffer[(region, 'Total')] = df[grp_msk].groupby('Publication year')['item_id'].nunique()
        buffer[(region, 'C&H include')] = df[df['incl_major'] & grp_msk].groupby('Publication year')['item_id'].nunique()
        buffer[(region, ' + impacts include')] = df[df['incl_major'] & df['incl_impacts'] & grp_msk].groupby('Publication year')['item_id'].nunique()
        buffer[(region, ' + attributable')] = df_locations[df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location'] & df_locations['grid_attributable'] & (df_locations[grouping] == region)].groupby('Publication year')['item_id'].nunique()
        buffer[(region, ' / not attributable')] = df_locations[df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location'] & ~df_locations['grid_attributable'] & (df_locations[grouping] == region)].groupby('Publication year')['item_id'].nunique()
    
    df_abs = pd.DataFrame(buffer).fillna(0).astype(int)
    display(df_abs)
    display(Markdown('**Proportion of attributable research per region**'))
    display((df_abs[regions].T / df_abs[('Global', ' + attributable')]*100).xs(' + attributable', level=1))
    display(Markdown('**Proportion of attributable research per region**'))
    display(pd.DataFrame({
        '2006–2015': df_abs.loc[2006:2015].sum() / df_abs.loc[2006:2015].sum()[ ('Global', ' + attributable')] * 100,
        '2016–2025': df_abs.loc[2016:2025].sum() / df_abs.loc[2016:2025].sum()[ ('Global', ' + attributable')] * 100,
        '2006–2025': df_abs.sum() / df_abs.sum()[ ('Global', ' + attributable')] * 100, 
        '2006–2015 (abs)': df_abs.loc[2006:2015].sum(),
        '2016–2025 (abs)': df_abs.loc[2016:2025].sum(),
        '2006–2025 (abs)': df_abs.sum(), 
    }).T.drop(columns=['Global']))

In [ ]:
a = df_totals.T.reset_index(level=0, drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 3), dpi=150)

for rn, r in a.iterrows():
    axes[0].plot(r.index, r.values, label=rn, ls=':' if rn != 'Total' else '-')
axes[0].legend()

prev = np.zeros(a.shape[1])
for rn in reversed(a.index):
    if rn == 'Total': continue
    r = a.loc[rn]
    axes[1].bar(r.index, r - prev, bottom=prev, label=rn, ls=':' if rn != 'Total' else '-')
    prev = r
axes[1].legend()

fig.savefig(OUTPUT_DIR / 'impacts_filters.png')
fig.savefig(OUTPUT_DIR / 'impacts_filters.pdf')
fig.savefig(OUTPUT_DIR / 'impacts_filters.svg')

## Exposed people

In [ ]:
# FAB4B8
# F768A0
# AD017D

for grp in [
    'Group (Lancet 2026)', 
    #'Group (WHO 2026)', 
    'Group (HDI 2026)', 
    'Group (HDI-2 2026)', 
    'Region (WorldBank 2026)', 
    'Income group (WorldBank 2026)',
    #'Lending category (WorldBank 2026)', 
    #'Continent (Name)'
]:
    display(Markdown(f'### {grp}'))
    df_exposed = pd.DataFrame({
        'Population (in M)': df_countries.groupby(grp)['Population (Lancet, 2025)'].sum() / 1000000,
        'Number of studies': df_locations[df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location'] & df_locations['grid_attributable']].groupby(grp)['item_id'].nunique(),
    })
    if grp in ORDERING:
        df_exposed = df_exposed.loc[ORDERING[grp]]
    df_exposed['Studies per million citizens exposed'] = df_exposed['Number of studies']  / df_exposed['Population (in M)']
    display(df_exposed)

    fig, axes = plt.subplots(1, 1, figsize=(7.5,5), dpi=150)
    axes = [axes]

    W = 0.35
    h1 = axes[0].barh(np.arange(len(df_exposed.index))+(W/2), df_exposed['Number of studies'], height=W, label='Number of studies', facecolor='#F768A0')
    h2 = axes[0].barh(np.arange(len(df_exposed.index))-(W/2), df_exposed['Population (in M)'], height=W, label='Population in million', facecolor='#AD017D')
    axes[0].set_yticks(range(df_exposed.shape[0]))
    axes[0].set_yticklabels(df_exposed.index)
    

    ax = axes[0].twiny()
    h3 = ax.barh(df_exposed.index, df_exposed['Studies per million citizens exposed'], label='Studies per million citizens', height=0.15, facecolor='#FAB4B8', hatch='\\')
    ax.set_ylabel('Studies per million citizens')
    #fig.legend(loc='upper right')
    axes[0].legend(handles=[h1,h2,h3])
    if len(axes)>1:
        axes[1].set_title('Studies per million citizens exposed', fontsize=10)
        axes[1].barh(df_exposed.index, df_exposed['Studies per million citizens exposed'], label='Studies per million citizens', height=0.5)
        axes[1].set(yticklabels=[])
    display(fig)
    
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.png')
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.pdf')
    fig.savefig(OUTPUT_DIR / f'affected_{grp}.svg')
    fig.clf()
    fig.clear()

## Sankeys

In [ ]:
df.drop(columns=['abstract', 'title', 'authors']).head()

In [ ]:
df_locations.head()

In [ ]:
ATTRIBUTABLE = {
    'driver|1': ('Changes in temperature', 'temp_da'),
    #'climateDrivers|2': 'Seasonal Change',
    'driver|3': ('Changes in precipitation', 'precip_da'),
    #'climateDrivers|4': 'Sea-level rise',
    #'climateDrivers|5': 'Climate Change (unspecified) ',
    #'climateDrivers|7': 'Other meteorological variables',
    #'climateDrivers|6': 'Changes in humidity'},
}

# 'driver',     'event',     'health',     'expose',    'attr',    'sector',     'keywords',
#tab = df_study_grid.merge(df.reset_index(drop=True), left_on='item_id', right_on='item_id', suffixes=('_x', ''))

#df_locations[df_locations['incl_major'] & df_locations['incl_impacts'] & df_locations['incl_location'] & df_locations['grid_attributable'] & (df_locations[grouping] == region)]

mask_base = df['incl_impacts'] & df['incl_location']# & (tab['is_land'] == True)
mask_attr = df['grid_attributable'] == True

lab1 = LABELS['driver']
lab2 = LABELS['health']

n_a = df[mask_base & mask_attr]['item_id'].nunique()
n_na = df[mask_base & ~mask_attr]['item_id'].nunique()

data = []
for i, label1 in enumerate(lab1.labels):
    name1 = label1.name
    key1 = label1.column
    if key1 not in df: continue
    print(name1, key1)
    m1 = (df[key1] >= 0.5) 
    n1 = df[m1]['openalex_id'].nunique()

    for j, label2 in enumerate(lab2.labels):
        name2 = label2.name
        key2 = label2.column
        if key2 not in df: continue
        m2 = (df[key2] >= 0.5)
        n2 = df[m2]['item_id'].nunique()
        if key1 in ATTRIBUTABLE:
            data.append([
                df[mask_base & mask_attr & m1 & m2]['item_id'].nunique(),
                f'Attributable ({n_a:,})',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])
            data.append([
                df[mask_base & ~mask_attr & m1 & m2]['item_id'].nunique(),
                f'Not attributable ({n_na:,})',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])
        else:
            data.append([
                df[mask_base & m1 & m2]['item_id'].nunique(),
                'NA',
                f'{name1} ({n1:,})',
                f'{name2} ({n2:,})',
            ])

sdf = pd.DataFrame(data, columns=['Weight', 'In area where trend is', lab1.name, lab2.name])

fig = plt.figure(figsize=(7.4, 7), dpi=150)
sankey(sdf, cmap=plt.get_cmap('viridis'), labels_size=8, titles_size=10)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / 'sankey.png')
fig.savefig(OUTPUT_DIR / 'sankey.svg')
fig.savefig(OUTPUT_DIR / 'sankey.pdf')